In [7]:
import pandas as pd
import os

def extract_best_results():
    """
    Extract best results from SARIMA, Prophet, and Ensemble models
    and save them to separate CSV files for LaTeX table creation.
    Uses RMSE as the primary metric to handle zero MAPE issues.
    """
    
    installations = ['3001084033', '3001449459', '3003858507', '3011373971', '3011504476']
    
    # Initialize dataframes for best results
    best_sarima_results = []
    best_prophet_results = []
    best_ensemble_results = []
    
    print("Extracting best results based on RMSE metric...")
    print("=" * 60)
    
    for installation in installations:
        print(f"\nProcessing installation: {installation}")
        
        # Process SARIMA results
        sarima_file = f"sarima_results_{installation}.csv"
        if os.path.exists(sarima_file):
            try:
                sarima_df = pd.read_csv(sarima_file)
                sarima_success = sarima_df[sarima_df['Status'] == 'Success']
                
                if len(sarima_success) > 0:
                    # Sort by RMSE and get the best result
                    best_sarima = sarima_success.sort_values('RMSE').iloc[0]
                    
                    best_sarima_row = {
                        'Installation': installation,
                        'p': int(best_sarima['p']),
                        'd': int(best_sarima['d']),
                        'q': int(best_sarima['q']),
                        'P': int(best_sarima['P']),
                        'D': int(best_sarima['D']),
                        'Q': int(best_sarima['Q']),
                        's': int(best_sarima['s']),
                        'RMSE': round(best_sarima['RMSE'], 4),
                        'MAE': round(best_sarima['MAE'], 4),
                        'MAPE': round(best_sarima['MAPE'], 4),
                        'WMAPE': round(best_sarima['WMAPE'], 4)
                    }
                    best_sarima_results.append(best_sarima_row)
                    print(f"  SARIMA: ({int(best_sarima['p'])},{int(best_sarima['d'])},{int(best_sarima['q'])})x({int(best_sarima['P'])},{int(best_sarima['D'])},{int(best_sarima['Q'])},12) - RMSE: {best_sarima['RMSE']:.4f}")
                else:
                    print(f"  SARIMA: No successful results found")
            except Exception as e:
                print(f"  SARIMA: Error processing - {e}")
        else:
            print(f"  SARIMA: File not found - {sarima_file}")
        
        # Process Prophet results
        prophet_file = f"prophet_results_{installation}.csv"
        if os.path.exists(prophet_file):
            try:
                prophet_df = pd.read_csv(prophet_file)
                prophet_success = prophet_df[prophet_df['Status'] == 'Success']
                
                if len(prophet_success) > 0:
                    # Sort by RMSE and get the best result
                    best_prophet = prophet_success.sort_values('RMSE').iloc[0]
                    
                    best_prophet_row = {
                        'Installation': installation,
                        'yearly_seasonality': int(best_prophet['yearly_seasonality']),
                        'RMSE': round(best_prophet['RMSE'], 4),
                        'MAE': round(best_prophet['MAE'], 4),
                        'MAPE': round(best_prophet['MAPE'], 4),
                        'WMAPE': round(best_prophet['WMAPE'], 4)
                    }
                    best_prophet_results.append(best_prophet_row)
                    print(f"  Prophet: yearly_seasonality={int(best_prophet['yearly_seasonality'])} - RMSE: {best_prophet['RMSE']:.4f}")
                else:
                    print(f"  Prophet: No successful results found")
            except Exception as e:
                print(f"  Prophet: Error processing - {e}")
        else:
            print(f"  Prophet: File not found - {prophet_file}")
    
    # Process Ensemble results (single file with all installations and methods)
    ensemble_file = "ensemble_results.csv"
    if os.path.exists(ensemble_file):
        try:
            ensemble_df = pd.read_csv(ensemble_file)
            print(f"  Ensemble file loaded: {len(ensemble_df)} rows")
            print(f"  Installations in ensemble file: {ensemble_df['Installation'].unique()}")
            
            for installation in installations:
                # Convert installation to string for comparison
                installation_str = str(installation)
                installation_data = ensemble_df[ensemble_df['Installation'].astype(str) == installation_str]
                
                print(f"  Installation {installation}: found {len(installation_data)} methods")
                
                if len(installation_data) > 0:
                    # Sort by RMSE and get the best result
                    best_ensemble = installation_data.sort_values('RMSE').iloc[0]
                    
                    # Handle missing Optimal_Weight for some methods
                    optimal_weight = best_ensemble['Optimal_Weight']
                    if pd.isna(optimal_weight):
                        optimal_weight = None
                    else:
                        optimal_weight = round(optimal_weight, 4)
                    
                    best_ensemble_row = {
                        'Installation': installation,
                        'Method': best_ensemble['Method'],
                        'RMSE': round(best_ensemble['RMSE'], 4),
                        'MAE': round(best_ensemble['MAE'], 4),
                        'MAPE': round(best_ensemble['MAPE'], 4),
                        'WMAPE': round(best_ensemble['WMAPE'], 4),
                        'Optimal_Weight': optimal_weight
                    }
                    best_ensemble_results.append(best_ensemble_row)
                    
                    weight_str = f"weight: {optimal_weight}" if optimal_weight is not None else "no weight"
                    print(f"    Best: {best_ensemble['Method']} ({weight_str}) - RMSE: {best_ensemble['RMSE']:.4f}")
                else:
                    print(f"    No results found for installation {installation}")
        except Exception as e:
            print(f"  Ensemble: Error processing - {e}")
    else:
        print(f"  Ensemble: File not found - {ensemble_file}")
    
    # Save results to CSV files
    print("\n" + "=" * 60)
    print("Saving results to CSV files...")
    
    # Save SARIMA results
    if best_sarima_results:
        sarima_df = pd.DataFrame(best_sarima_results)
        sarima_df.to_csv('best_sarima_results.csv', index=False)
        print(f"✓ Saved {len(best_sarima_results)} SARIMA results to 'best_sarima_results.csv'")
    else:
        print("✗ No SARIMA results to save")
    
    # Save Prophet results
    if best_prophet_results:
        prophet_df = pd.DataFrame(best_prophet_results)
        prophet_df.to_csv('best_prophet_results.csv', index=False)
        print(f"✓ Saved {len(best_prophet_results)} Prophet results to 'best_prophet_results.csv'")
    else:
        print("✗ No Prophet results to save")
    
    # Save Ensemble results
    if best_ensemble_results:
        ensemble_df = pd.DataFrame(best_ensemble_results)
        ensemble_df.to_csv('best_ensemble_results.csv', index=False)
        print(f"✓ Saved {len(best_ensemble_results)} Ensemble results to 'best_ensemble_results.csv'")
    else:
        print("✗ No Ensemble results to save")
    
    print("\nExtraction complete!")
    
    # Print summary statistics
    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    
    if best_sarima_results:
        sarima_df = pd.DataFrame(best_sarima_results)
        print(f"SARIMA - Average RMSE: {sarima_df['RMSE'].mean():.4f}")
        print(f"SARIMA - Average MAPE: {sarima_df['MAPE'].mean():.4f}")
    
    if best_prophet_results:
        prophet_df = pd.DataFrame(best_prophet_results)
        print(f"Prophet - Average RMSE: {prophet_df['RMSE'].mean():.4f}")
        print(f"Prophet - Average MAPE: {prophet_df['MAPE'].mean():.4f}")
    
    if best_ensemble_results:
        ensemble_df = pd.DataFrame(best_ensemble_results)
        print(f"Ensemble - Average RMSE: {ensemble_df['RMSE'].mean():.4f}")
        print(f"Ensemble - Average MAPE: {ensemble_df['MAPE'].mean():.4f}")

def create_detailed_ensemble_analysis():
    """
    Create a detailed analysis of all ensemble methods for each installation
    """
    print("\n" + "=" * 60)
    print("Creating detailed ensemble analysis...")
    
    try:
        ensemble_df = pd.read_csv('ensemble_results.csv')
        
        # Create a detailed table with all methods
        detailed_analysis = []
        
        installations = ['3001084033', '3001449459', '3003858507', '3011373971', '3011504476']
        methods = ['SARIMA', 'Prophet', 'Simple_Voting', 'Weighted_Voting', 'Random_Forest']
        
        for installation in installations:
            installation_str = str(installation)
            installation_data = ensemble_df[ensemble_df['Installation'].astype(str) == installation_str]
            
            for method in methods:
                method_data = installation_data[installation_data['Method'] == method]
                
                if len(method_data) > 0:
                    method_row = method_data.iloc[0]
                    
                    # Handle missing Optimal_Weight
                    optimal_weight = method_row['Optimal_Weight']
                    if pd.isna(optimal_weight):
                        optimal_weight = None
                    else:
                        optimal_weight = round(optimal_weight, 4)
                    
                    row = {
                        'Installation': installation,
                        'Method': method,
                        'RMSE': round(method_row['RMSE'], 4),
                        'MAE': round(method_row['MAE'], 4),
                        'MAPE': round(method_row['MAPE'], 4),
                        'WMAPE': round(method_row['WMAPE'], 4),
                        'Optimal_Weight': optimal_weight
                    }
                    detailed_analysis.append(row)
        
        detailed_df = pd.DataFrame(detailed_analysis)
        detailed_df.to_csv('detailed_ensemble_analysis.csv', index=False)
        print("✓ Saved detailed ensemble analysis to 'detailed_ensemble_analysis.csv'")
        
        # Also create a pivot table for easier LaTeX formatting
        pivot_data = []
        for installation in installations:
            row = {'Installation': installation}
            installation_str = str(installation)
            installation_data = ensemble_df[ensemble_df['Installation'].astype(str) == installation_str]
            
            for method in methods:
                method_data = installation_data[installation_data['Method'] == method]
                if len(method_data) > 0:
                    method_row = method_data.iloc[0]
                    row[f'{method}_RMSE'] = round(method_row['RMSE'], 4)
                    row[f'{method}_MAPE'] = round(method_row['MAPE'], 4)
                    
                    # Add weight for Weighted_Voting
                    if method == 'Weighted_Voting' and not pd.isna(method_row['Optimal_Weight']):
                        row[f'{method}_Weight'] = round(method_row['Optimal_Weight'], 4)
                else:
                    row[f'{method}_RMSE'] = 'N/A'
                    row[f'{method}_MAPE'] = 'N/A'
            
            pivot_data.append(row)
        
        pivot_df = pd.DataFrame(pivot_data)
        pivot_df.to_csv('ensemble_pivot_table.csv', index=False)
        print("✓ Saved ensemble pivot table to 'ensemble_pivot_table.csv'")
        
        return detailed_df, pivot_df
        
    except Exception as e:
        print(f"Error creating detailed ensemble analysis: {e}")
        return None, None

def create_comparison_table():
    """
    Create a comparison table with all models for LaTeX
    """
    print("\n" + "=" * 60)
    print("Creating comparison table...")
    
    try:
        # Load the best results
        sarima_df = pd.read_csv('best_sarima_results.csv') if os.path.exists('best_sarima_results.csv') else pd.DataFrame()
        prophet_df = pd.read_csv('best_prophet_results.csv') if os.path.exists('best_prophet_results.csv') else pd.DataFrame()
        ensemble_df = pd.read_csv('best_ensemble_results.csv') if os.path.exists('best_ensemble_results.csv') else pd.DataFrame()
        
        # Create comparison dataframe
        comparison_data = []
        
        installations = ['3001084033', '3001449459', '3003858507', '3011373971', '3011504476']
        
        for installation in installations:
            row = {'Installation': installation}
            
            # SARIMA data
            if not sarima_df.empty:
                sarima_row = sarima_df[sarima_df['Installation'].astype(str) == str(installation)]
                if len(sarima_row) > 0:
                    sarima_row = sarima_row.iloc[0]
                    row['SARIMA_RMSE'] = round(sarima_row['RMSE'], 4)
                    row['SARIMA_MAPE'] = round(sarima_row['MAPE'], 4)
                    row['SARIMA_Params'] = f"({int(sarima_row['p'])},{int(sarima_row['d'])},{int(sarima_row['q'])})x({int(sarima_row['P'])},{int(sarima_row['D'])},{int(sarima_row['Q'])},12)"
                else:
                    row['SARIMA_RMSE'] = 'N/A'
                    row['SARIMA_MAPE'] = 'N/A'
                    row['SARIMA_Params'] = 'N/A'
            else:
                row['SARIMA_RMSE'] = 'N/A'
                row['SARIMA_MAPE'] = 'N/A'
                row['SARIMA_Params'] = 'N/A'
            
            # Prophet data
            if not prophet_df.empty:
                prophet_row = prophet_df[prophet_df['Installation'].astype(str) == str(installation)]
                if len(prophet_row) > 0:
                    prophet_row = prophet_row.iloc[0]
                    row['Prophet_RMSE'] = round(prophet_row['RMSE'], 4)
                    row['Prophet_MAPE'] = round(prophet_row['MAPE'], 4)
                    row['Prophet_Seasonality'] = int(prophet_row['yearly_seasonality'])
                else:
                    row['Prophet_RMSE'] = 'N/A'
                    row['Prophet_MAPE'] = 'N/A'
                    row['Prophet_Seasonality'] = 'N/A'
            else:
                row['Prophet_RMSE'] = 'N/A'
                row['Prophet_MAPE'] = 'N/A'
                row['Prophet_Seasonality'] = 'N/A'
            
            # Ensemble data
            if not ensemble_df.empty:
                ensemble_row = ensemble_df[ensemble_df['Installation'].astype(str) == str(installation)]
                if len(ensemble_row) > 0:
                    ensemble_row = ensemble_row.iloc[0]
                    row['Ensemble_RMSE'] = round(ensemble_row['RMSE'], 4)
                    row['Ensemble_MAPE'] = round(ensemble_row['MAPE'], 4)
                    row['Ensemble_Method'] = ensemble_row['Method']
                    
                    # Handle missing Optimal_Weight
                    if pd.isna(ensemble_row['Optimal_Weight']):
                        row['Ensemble_Weight'] = 'N/A'
                    else:
                        row['Ensemble_Weight'] = round(ensemble_row['Optimal_Weight'], 4)
                else:
                    row['Ensemble_RMSE'] = 'N/A'
                    row['Ensemble_MAPE'] = 'N/A'
                    row['Ensemble_Method'] = 'N/A'
                    row['Ensemble_Weight'] = 'N/A'
            else:
                row['Ensemble_RMSE'] = 'N/A'
                row['Ensemble_MAPE'] = 'N/A'
                row['Ensemble_Method'] = 'N/A'
                row['Ensemble_Weight'] = 'N/A'
            
            comparison_data.append(row)
        
        comparison_df = pd.DataFrame(comparison_data)
        comparison_df.to_csv('model_comparison.csv', index=False)
        print("✓ Saved model comparison to 'model_comparison.csv'")
        
        return comparison_df
        
    except Exception as e:
        print(f"Error creating comparison table: {e}")
        return None

def create_latex_ready_tables():
    """
    Create simplified tables ready for LaTeX with proper formatting
    """
    print("\n" + "=" * 60)
    print("Creating LaTeX-ready tables...")
    
    try:
        # Load ensemble data
        ensemble_df = pd.read_csv('ensemble_results.csv')
        
        # Table 1: Best model parameters (if SARIMA/Prophet files exist)
        best_params_data = []
        
        sarima_df = pd.read_csv('best_sarima_results.csv') if os.path.exists('best_sarima_results.csv') else pd.DataFrame()
        prophet_df = pd.read_csv('best_prophet_results.csv') if os.path.exists('best_prophet_results.csv') else pd.DataFrame()
        
        installations = ['3001084033', '3001449459', '3003858507', '3011373971', '3011504476']
        
        for installation in installations:
            row = {'Installation': installation}
            
            # SARIMA parameters
            if not sarima_df.empty:
                sarima_row = sarima_df[sarima_df['Installation'].astype(str) == str(installation)]
                if len(sarima_row) > 0:
                    sr = sarima_row.iloc[0]
                    row['SARIMA_Parameters'] = f"({int(sr['p'])},{int(sr['d'])},{int(sr['q'])})x({int(sr['P'])},{int(sr['D'])},{int(sr['Q'])},12)"
                    row['SARIMA_RMSE'] = f"{sr['RMSE']:.2f}"
                    row['SARIMA_MAPE'] = f"{sr['MAPE']*100:.2f}\\%"
                else:
                    row['SARIMA_Parameters'] = 'N/A'
                    row['SARIMA_RMSE'] = 'N/A'
                    row['SARIMA_MAPE'] = 'N/A'
            else:
                row['SARIMA_Parameters'] = 'N/A'
                row['SARIMA_RMSE'] = 'N/A'
                row['SARIMA_MAPE'] = 'N/A'
            
            # Prophet parameters
            if not prophet_df.empty:
                prophet_row = prophet_df[prophet_df['Installation'].astype(str) == str(installation)]
                if len(prophet_row) > 0:
                    pr = prophet_row.iloc[0]
                    row['Prophet_Seasonality'] = f"{int(pr['yearly_seasonality'])}"
                    row['Prophet_RMSE'] = f"{pr['RMSE']:.2f}"
                    row['Prophet_MAPE'] = f"{pr['MAPE']*100:.2f}\\%"
                else:
                    row['Prophet_Seasonality'] = 'N/A'
                    row['Prophet_RMSE'] = 'N/A'
                    row['Prophet_MAPE'] = 'N/A'
            else:
                row['Prophet_Seasonality'] = 'N/A'
                row['Prophet_RMSE'] = 'N/A'
                row['Prophet_MAPE'] = 'N/A'
            
            best_params_data.append(row)
        
        if best_params_data:
            params_df = pd.DataFrame(best_params_data)
            params_df.to_csv('latex_model_parameters.csv', index=False)
            print("✓ Saved LaTeX model parameters to 'latex_model_parameters.csv'")
        
        # Table 2: Ensemble methods comparison
        latex_ensemble_data = []
        methods = ['SARIMA', 'Prophet', 'Simple_Voting', 'Weighted_Voting', 'Random_Forest']
        
        for installation in installations:
            row = {'Installation': installation}
            installation_str = str(installation)
            installation_data = ensemble_df[ensemble_df['Installation'].astype(str) == installation_str]
            
            for method in methods:
                method_data = installation_data[installation_data['Method'] == method]
                if len(method_data) > 0:
                    mr = method_data.iloc[0]
                    row[f'{method}_RMSE'] = f"{mr['RMSE']:.2f}"
                    row[f'{method}_MAPE'] = f"{mr['MAPE']*100:.2f}\\%"
                    
                    if method == 'Weighted_Voting' and not pd.isna(mr['Optimal_Weight']):
                        row[f'{method}_Weight'] = f"{mr['Optimal_Weight']:.3f}"
                else:
                    row[f'{method}_RMSE'] = 'N/A'
                    row[f'{method}_MAPE'] = 'N/A'
            
            latex_ensemble_data.append(row)
        
        ensemble_latex_df = pd.DataFrame(latex_ensemble_data)
        ensemble_latex_df.to_csv('latex_ensemble_comparison.csv', index=False)
        print("✓ Saved LaTeX ensemble comparison to 'latex_ensemble_comparison.csv'")
        
        print("LaTeX-ready tables created successfully!")
        
    except Exception as e:
        print(f"Error creating LaTeX-ready tables: {e}")

if __name__ == "__main__":
    # Extract best results
    extract_best_results()
    
    # Create comparison table
    create_comparison_table()
    
    # Create detailed ensemble analysis
    create_detailed_ensemble_analysis()
    
    # Create LaTeX-ready tables
    create_latex_ready_tables()
    
    print("\n" + "=" * 80)
    print("ALL FILES CREATED SUCCESSFULLY!")
    print("=" * 80)
    print("\nFiles created:")
    print("- best_sarima_results.csv (Best SARIMA parameters per installation)")
    print("- best_prophet_results.csv (Best Prophet parameters per installation)")
    print("- best_ensemble_results.csv (Best ensemble method per installation)")
    print("- model_comparison.csv (Side-by-side comparison of all models)")
    print("- detailed_ensemble_analysis.csv (All ensemble methods and metrics)")
    print("- ensemble_pivot_table.csv (Ensemble results in pivot format)")
    print("- latex_model_parameters.csv (LaTeX-ready model parameters table)")
    print("- latex_ensemble_comparison.csv (LaTeX-ready ensemble comparison table)")
    print("\nThe files are ready for LaTeX table creation!")

Extracting best results based on RMSE metric...

Processing installation: 3001084033
  SARIMA: (10,1,8)x(1,0,1,12) - RMSE: 656.8730
  Prophet: yearly_seasonality=12 - RMSE: 929.1025

Processing installation: 3001449459
  SARIMA: (9,1,7)x(1,0,1,12) - RMSE: 225.3050
  Prophet: yearly_seasonality=12 - RMSE: 336.9158

Processing installation: 3003858507
  SARIMA: (9,1,10)x(1,0,0,12) - RMSE: 435.0292
  Prophet: yearly_seasonality=12 - RMSE: 728.4678

Processing installation: 3011373971
  SARIMA: (10,1,9)x(1,0,1,12) - RMSE: 114.0750
  Prophet: yearly_seasonality=10 - RMSE: 163.7356

Processing installation: 3011504476
  SARIMA: (10,0,10)x(1,0,1,12) - RMSE: 233.4798
  Prophet: yearly_seasonality=12 - RMSE: 641.5376
  Ensemble file loaded: 25 rows
  Installations in ensemble file: [3001084033 3001449459 3003858507 3011373971 3011504476]
  Installation 3001084033: found 5 methods
    Best: Weighted_Voting (weight: 0.5858) - RMSE: 580.3857
  Installation 3001449459: found 5 methods
    Best: SAR